# Kapitel 20.4 - Asyncio Praxis: Timeouts, Cancellation, Robustheit

# Lernziele

- Timeouts in Async-Workflows planen
- Tasks kontrolliert abbrechen
- Teilfehler robust behandeln

# Voraussetzungen

- Kapitel 20.3

# Theorie

Produktionscode braucht Fehlertoleranz: nicht alle Tasks werden erfolgreich sein.
Timeouts und Cancellation verhindern, dass Systeme unendlich warten.

# Erklaerung

- Timeout: begrenzte Wartezeit
- Cancellation: bewusstes Abbrechen einer Aufgabe
- Retry: kontrollierter Wiederholungsversuch

# Syntax

```python
task = asyncio.create_task(coro())
task.cancel()
await asyncio.wait_for(coro(), timeout=1.5)
```

# Merke

Cancellation ist kein Fehler, sondern ein normaler Kontrollfluss im Async-Kontext.

# Parameter

- `timeout` in Sekunden
- `return_exceptions=True` fuer Sammelverarbeitung

# Rueckgabewert

Tasks liefern Werte oder Exceptions. Beide muessen explizit ausgewertet werden.

In [ ]:
import asyncio

async def worker(name, dauer):
    await asyncio.sleep(dauer)
    return f'{name} ok'

async def demo_timeout():
    try:
        result = await asyncio.wait_for(worker('Langsam', 2), timeout=1)
        print(result)
    except asyncio.TimeoutError:
        print('Timeout bei Langsam')

asyncio.run(demo_timeout())

In [ ]:
async def demo_cancellation():
    task = asyncio.create_task(worker('Abbrechen', 5))
    await asyncio.sleep(1)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print('Task wurde sauber abgebrochen')

asyncio.run(demo_cancellation())

In [ ]:
async def maybe_fail(i):
    await asyncio.sleep(0.1)
    if i % 4 == 0:
        raise ValueError(f'Fehler bei {i}')
    return i * 10

async def demo_batch():
    tasks = [maybe_fail(i) for i in range(1, 9)]
    ergebnisse = await asyncio.gather(*tasks, return_exceptions=True)
    print(ergebnisse)

asyncio.run(demo_batch())

# Praxisbeispiel

Baue einen asynchronen Batch-Prozessor mit Timeout pro Job und Fehlerreport.

# Haeufige Fehler

1. Exception in Task ignorieren.
2. Kein Timeout fuer externe Abhaengigkeiten setzen.
3. Cancellation ohne Cleanup.

# Best Practice

- Einheitliche Fehlerstruktur pro Task.
- Timeouts zentral konfigurieren.
- Abgebrochene Tasks sauber protokollieren.

# Tipp

Definiere fuer kritische APIs Default-Timeouts und uebersteuerbare Konfigurationswerte.

# Uebung

Implementiere einen Retry-Mechanismus mit maximal 3 Versuchen bei Timeout.

# Loesung

Nutze eine Schleife mit Zaehler und `try/except asyncio.TimeoutError`, danach ggf. Rueckgabe eines Fehlerobjekts.

# Zusammenfassung

Du kannst Asyncio-Aufgaben jetzt robust steuern: Timeout, Cancellation und Fehleraggregation.

# Weiterfuehrende Links

- asyncio task cancellation

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

Race Condition
Critical Section
Event Loop
Backpressure
Cancellation
Timeout Budget

In [ ]:
# asyncio Timeout-Guard
import asyncio
async def slow_job():
    await asyncio.sleep(2)
    return "done"
async def main():
    try:
        print(await asyncio.wait_for(slow_job(), timeout=1.0))
    except asyncio.TimeoutError:
        print("timeout")
asyncio.run(main())

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.